In [7]:
!pip install google-play-scraper

In [8]:
from google_play_scraper import reviews, app
import pandas as pd

app_package = 'com.whatsapp'
result, _ = reviews(
    app_package,
    lang='id',
    country='id',
    count=10000
)

df = pd.DataFrame(result)
df.to_csv('playstore_reviews.csv', index=False)
print(f"{len(result)} data Saved")


10000 data Saved


In [5]:
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# dowload data buat processing
nltk.download('stopwords')
nltk.download('punkt_tab')

file_path = 'playstore_reviews.csv'
df = pd.read_csv(file_path)

# data processing
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  #URLs
    text = re.sub(r'\W', ' ', text)  #non-alphanumeric characters
    text = re.sub(r'\s+', ' ', text)  #multiple spaces
    text = re.sub(r'\d', '', text)  #digits
    text = text.lower().strip()  #lowercase dan strip
    return text

df['cleaned_review'] = df['content'].apply(clean_text)

# Tokenization and Stopword Removal
stop_words = set(stopwords.words('indonesian'))#common word

df['tokenized_review'] = df['cleaned_review'].apply(
    lambda x: ' '.join([word for word in word_tokenize(x) if word not in stop_words])
)
#saya suka fitur yang menarik => saya    suka    fitur    menarik    => saya suka fitur menarik
#label
df['sentiment'] = df['score'].apply(lambda x: 1 if x >= 4 else 0)

# Split data
X = df['tokenized_review'] #feature
y = df['sentiment'] #label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [9]:
# Naive Bayes Model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

# SVM Model
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)


# Evaluation
print("Naive Bayes Performance")
print(classification_report(y_test, nb_preds, output_dict=False))  # Only class-level metrics
print(f"Accuracy: {accuracy_score(y_test, nb_preds)}")

print("\nSVM Performance")
print(classification_report(y_test, svm_preds, output_dict=False))  # Only class-level metrics
print(f"Accuracy: {accuracy_score(y_test, svm_preds)}")


Naive Bayes Performance
              precision    recall  f1-score   support

           0       0.72      0.58      0.64       664
           1       0.81      0.89      0.85      1336

    accuracy                           0.79      2000
   macro avg       0.76      0.73      0.74      2000
weighted avg       0.78      0.79      0.78      2000

Accuracy: 0.7855

SVM Performance
              precision    recall  f1-score   support

           0       0.71      0.58      0.64       664
           1       0.81      0.88      0.84      1336

    accuracy                           0.78      2000
   macro avg       0.76      0.73      0.74      2000
weighted avg       0.78      0.78      0.78      2000

Accuracy: 0.783
